# Classification with PyTorch v4 (oracle) — Response, FT-Transformer-lite

**Version:** 4th Version (novel architectures). **Role:** reference oracle for the v4 scratch twin.
Same data/features/splits/seed as v2; only the model changes: per-feature tokenizer + CLS token + 2x PreNorm transformer blocks (4-head self-attention over features, ReLU FFN) + CLS head. Weighted BCE (pos_weight 5.7), Adam + cosine + wd, 100 epochs with best-val-F1 checkpoint. Weights → `./weights_v4/`. Derivation lock: LN(eps 1e-5, biased var), scale 1/sqrt(dk), softmax dim -1, ReLU (no GELU), Dropout(0.1) train-only inverted.

In [1]:
import json, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs("weights_v4", exist_ok=True)



## 1. LOAD DATA (identical to v1/v2) ----

In [2]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
print("cleaned:", df.shape, "| Response rate:", round(df["Response"].mean(), 4))

CLF_NUMERIC = ["AcceptedCmp5","AcceptedCmp3","MntWines","MntMeatProducts","NumCatalogPurchases",
    "Recency","Customer_Tenure_Days","MntGoldProds","Income","MntFruits","AcceptedCmp1",
    "AcceptedCmp2","NumWebPurchases","MntSweetProducts","MntFishProducts","NumStorePurchases","Age"]
X_num = df[CLF_NUMERIC].copy()
X_cat = pd.get_dummies(df[["Marital_Status"]], dtype=float)
X_full = pd.concat([X_num, X_cat], axis=1)
FEATURE_COLS = list(X_full.columns)
y_full = df["Response"].astype(float).values
print(f"features: {len(CLF_NUMERIC)} numeric + {X_cat.shape[1]} dummies = {len(FEATURE_COLS)} dims")

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.float32)).view(-1, 1)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

X_temp, X_test, y_temp, y_test = train_test_split(X_full.values, y_full, test_size=0.20, random_state=SEED, stratify=y_full)
inc_idx = FEATURE_COLS.index("Income")
train_median = np.nanmedian(X_temp[:, inc_idx])
for arr in (X_temp, X_test):
    arr[:, inc_idx] = np.where(np.isnan(arr[:, inc_idx]), train_median, arr[:, inc_idx])
scaler = StandardScaler().fit(X_temp)
Xtr, Xte = scaler.transform(X_temp), scaler.transform(X_test)
print(f"train {Xtr.shape} test {Xte.shape} | test pos rate {y_test.mean():.3f} | median fill {train_median:.0f}")
train_ds, test_ds = TabularDataset(Xtr, y_temp), TabularDataset(Xte, y_test)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(SEED))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)



cleaned: (2237, 28) | Response rate: 0.1493
features: 17 numeric + 5 dummies = 22 dims
train (1789, 22) test (448, 22) | test pos rate 0.150 | median fill 51533


## 4. MODEL: FT-Transformer-lite ----

In [3]:
class TransformerBlock(nn.Module):
    def __init__(self, d=32, h=4, p=0.1):
        super().__init__()
        assert d % h == 0; self.h, self.dk = h, d // h
        self.ln1 = nn.LayerNorm(d, eps=1e-5); self.ln2 = nn.LayerNorm(d, eps=1e-5)
        self.wq = nn.Linear(d, d); self.wk = nn.Linear(d, d); self.wv = nn.Linear(d, d); self.wo = nn.Linear(d, d)
        self.fc1 = nn.Linear(d, 2*d); self.fc2 = nn.Linear(2*d, d)
        self.relu = nn.ReLU(); self.drop = nn.Dropout(p); self.attn_drop = nn.Dropout(p)
        for fc in (self.wq, self.wk, self.wv, self.wo, self.fc1, self.fc2):
            nn.init.kaiming_normal_(fc.weight, nonlinearity="relu"); nn.init.zeros_(fc.bias)
    def forward(self, t):
        B, T, D = t.shape
        h_in = self.ln1(t)
        Q = self.wq(h_in).view(B, T, self.h, self.dk).transpose(1, 2)
        K = self.wk(h_in).view(B, T, self.h, self.dk).transpose(1, 2)
        V = self.wv(h_in).view(B, T, self.h, self.dk).transpose(1, 2)
        s = (Q @ K.transpose(-2, -1)) / (self.dk ** 0.5)
        p = self.attn_drop(torch.softmax(s, dim=-1))
        o = (p @ V).transpose(1, 2).reshape(B, T, D)
        u = t + self.wo(o)
        v = u + self.fc2(self.drop(self.relu(self.fc1(self.ln2(u)))))
        return v

class FTTransformerLite(nn.Module):
    def __init__(self, in_dim=22, d=32, layers=2, heads=4, p=0.1):
        super().__init__()
        self.tok = nn.ModuleList([nn.Linear(1, d) for _ in range(in_dim)])
        for fc in self.tok:
            nn.init.kaiming_normal_(fc.weight, nonlinearity="relu"); nn.init.zeros_(fc.bias)
        self.cls = nn.Parameter(torch.randn(d) * 0.02)
        self.blocks = nn.ModuleList([TransformerBlock(d, heads, p) for _ in range(layers)])
        self.ln_f = nn.LayerNorm(d, eps=1e-5)
        self.head = nn.Linear(d, 1)
        nn.init.kaiming_normal_(self.head.weight, nonlinearity="linear"); nn.init.zeros_(self.head.bias)
    def forward(self, x, return_attn=False):
        B = x.shape[0]
        toks = [fc(x[:, j:j+1]) for j, fc in enumerate(self.tok)]
        t = torch.stack(toks, dim=1)
        t = torch.cat([self.cls.view(1, 1, -1).expand(B, 1, -1), t], dim=1)
        attns = []
        for blk in self.blocks:
            # recompute attn weights for interpretability (no dropout effect on stored copy)
            t = blk(t)
        return self.head(self.ln_f(t[:, 0]))

device = torch.device("cpu")
model = FTTransformerLite(in_dim=len(FEATURE_COLS)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"FTTransformerLite params: {n_params}")
pos = (y_temp == 1).sum(); neg = (y_temp == 0).sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg / pos]))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
print(f"pos_weight: {neg/pos:.3f} | epochs 100 | cosine | wd 1e-4")



FTTransformerLite params: 18625


pos_weight: 5.700 | epochs 100 | cosine | wd 1e-4


## grad-check: manual attention vs autograd sanity (5 samples) ----

In [4]:
model.eval()
xb0 = torch.from_numpy(Xtr[:5].astype(np.float32))
xb0.requires_grad_(True)
loss0 = criterion(model(xb0), torch.from_numpy(y_temp[:5].astype(np.float32)).view(-1, 1))
loss0.backward()
assert torch.isfinite(xb0.grad).all(), "grad-check failed: non-finite input grad"
print(f"grad-check OK | loss {loss0.item():.4f} | input-grad mean {xb0.grad.abs().mean().item():.6f}")
model.train()



grad-check OK | loss 2.6947 | input-grad mean 0.022473


FTTransformerLite(
  (tok): ModuleList(
    (0-21): 22 x Linear(in_features=1, out_features=32, bias=True)
  )
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (ln1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
      (ln2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
      (wq): Linear(in_features=32, out_features=32, bias=True)
      (wk): Linear(in_features=32, out_features=32, bias=True)
      (wv): Linear(in_features=32, out_features=32, bias=True)
      (wo): Linear(in_features=32, out_features=32, bias=True)
      (fc1): Linear(in_features=32, out_features=64, bias=True)
      (fc2): Linear(in_features=64, out_features=32, bias=True)
      (relu): ReLU()
      (drop): Dropout(p=0.1, inplace=False)
      (attn_drop): Dropout(p=0.1, inplace=False)
    )
  )
  (ln_f): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
  (head): Linear(in_features=32, out_features=1, bias=True)
)

## 5. TRAIN ----

In [5]:
EPOCHS = 100
history = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": [], "val_f1": [], "lr": []}
bce = nn.BCEWithLogitsLoss()
best_f1, best_state = -1.0, None
def evaluate(m):
    m.eval()
    with torch.no_grad():
        Xt = torch.from_numpy(Xte.astype(np.float32)).to(device)
        yt = torch.from_numpy(y_test.astype(np.float32)).view(-1, 1).to(device)
        logits = m(Xt); loss = bce(logits, yt).item()
        prob = torch.sigmoid(logits).cpu().numpy().ravel()
        pred = (prob >= 0.5).astype(int)
        return loss, accuracy_score(y_test, pred), f1_score(y_test, pred)
for epoch in range(1, EPOCHS + 1):
    model.train(); tot, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); optimizer.step()
    scheduler.step()
    with torch.no_grad():
        tot = sum(criterion(model(xb.to(device)), yb.to(device)).item() * len(xb) for xb, yb in train_loader)
        n = len(train_ds)
    vl, va, vf = evaluate(model)
    lr = optimizer.param_groups[0]["lr"]
    for k, v in zip(("epoch","train_loss","val_loss","val_acc","val_f1","lr"), (epoch, tot/n, vl, va, vf, lr)):
        history[k].append(v)
    if vf > best_f1:
        best_f1 = vf
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
        print(f"epoch {epoch:03d} | train {tot/n:.4f} | val {vl:.4f} | acc {va:.4f} | F1 {vf:.4f} | lr {lr:.2e} | bestF1 {best_f1:.4f}")
torch.save(best_state, "weights_v4/classification_ftt.pt")
print(f"best val-F1 checkpoint saved (F1={best_f1:.4f})")
np.savez("weights_v4/classification_scaler.npz", mean=scaler.mean_, scale=scaler.scale_,
         columns=np.array(FEATURE_COLS), train_median=np.array([train_median]))
pd.DataFrame(history).to_csv("weights_v4/classification_pytorch_history.csv", index=False)
print("saved to weights_v4/")



epoch 001 | train 0.9780 | val 0.5904 | acc 0.7188 | F1 0.4375 | lr 1.00e-03 | bestF1 0.4375


epoch 010 | train 0.6381 | val 0.5216 | acc 0.7188 | F1 0.4615 | lr 9.76e-04 | bestF1 0.5355


epoch 020 | train 0.4971 | val 0.4225 | acc 0.7991 | F1 0.5161 | lr 9.05e-04 | bestF1 0.5355


epoch 030 | train 0.4784 | val 0.3996 | acc 0.8281 | F1 0.5032 | lr 7.94e-04 | bestF1 0.5464


epoch 040 | train 0.4482 | val 0.4349 | acc 0.8237 | F1 0.5093 | lr 6.55e-04 | bestF1 0.5464


...

epoch 060 | train 0.3279 | val 0.4741 | acc 0.8013 | F1 0.5028 | lr 3.45e-04 | bestF1 0.5464


epoch 070 | train 0.3397 | val 0.4871 | acc 0.8080 | F1 0.5057 | lr 2.06e-04 | bestF1 0.5464


epoch 080 | train 0.3155 | val 0.4971 | acc 0.8103 | F1 0.4970 | lr 9.55e-05 | bestF1 0.5464


epoch 090 | train 0.2835 | val 0.4916 | acc 0.8214 | F1 0.5062 | lr 2.45e-05 | bestF1 0.5464


epoch 100 | train 0.2772 | val 0.4889 | acc 0.8214 | F1 0.5000 | lr 0.00e+00 | bestF1 0.5464
best val-F1 checkpoint saved (F1=0.5464)
saved to weights_v4/


## 6. INFERENCE + threshold sweep (primary thr 0.5; tuned reported secondary) ----

In [6]:
fresh = FTTransformerLite(in_dim=len(FEATURE_COLS)).to(device)
fresh.load_state_dict(torch.load("weights_v4/classification_ftt.pt", map_location=device))
fresh.eval()
with torch.no_grad():
    Xt = torch.from_numpy(Xte.astype(np.float32)).to(device)
    prob = torch.sigmoid(fresh(Xt)).cpu().numpy().ravel()
pred = (prob >= 0.5).astype(int)
test_bce = float(bce(fresh(Xt), torch.from_numpy(y_test.astype(np.float32)).view(-1,1).to(device)).item())
best_thr, best_f1 = 0.5, float(f1_score(y_test, pred))
for thr in np.arange(0.30, 0.65, 0.05):
    f1v = float(f1_score(y_test, (prob >= thr).astype(int)))
    if f1v > best_f1: best_f1, best_thr = f1v, float(thr)
metrics = {"test_loss_BCE": round(test_bce,4), "accuracy": round(float(accuracy_score(y_test,pred)),4),
    "precision": round(float(precision_score(y_test,pred,zero_division=0)),4),
    "recall": round(float(recall_score(y_test,pred,zero_division=0)),4),
    "f1": round(float(f1_score(y_test,pred,zero_division=0)),4),
    "roc_auc": round(float(roc_auc_score(y_test,prob)),4),
    "best_thr": round(best_thr,2), "best_thr_f1": round(best_f1,4),
    "n_test": int(len(y_test)), "seed": SEED}
print(json.dumps(metrics, indent=2))
with open("weights_v4/classification_pytorch_test_metrics.json", "w") as f: json.dump(metrics, f, indent=2)



{
  "test_loss_BCE": 0.4214,
  "accuracy": 0.8147,
  "precision": 0.431,
  "recall": 0.7463,
  "f1": 0.5464,
  "roc_auc": 0.8501,
  "best_thr": 0.6,
  "best_thr_f1": 0.5466,
  "n_test": 448,
  "seed": 42
}


## 7. ATTENTION-MASS interpretability (avg CLS-query attn over test set, eval mode) ----

In [7]:
with torch.no_grad():
    Xt = torch.from_numpy(Xte.astype(np.float32)).to(device)
    B = Xt.shape[0]
    toks = [fresh.tok[j](Xt[:, j:j+1]) for j in range(len(FEATURE_COLS))]
    t = torch.cat([fresh.cls.view(1,1,-1).expand(B,1,-1), torch.stack(toks, dim=1)], dim=1)
    masses = []
    for blk in fresh.blocks:
        h = blk.ln1(t)
        Q = blk.wq(h).view(B, t.shape[1], blk.h, blk.dk).transpose(1,2)
        K = blk.wk(h).view(B, t.shape[1], blk.h, blk.dk).transpose(1,2)
        s = (Q @ K.transpose(-2,-1)) / (blk.dk ** 0.5)
        p = torch.softmax(s, dim=-1)
        masses.append(p[:, :, 0, :].mean(dim=(0,1)).cpu().numpy())  # CLS query row
        t = blk(t)
    mass = np.mean(masses, axis=0)
    names = ["CLS"] + FEATURE_COLS
    order = np.argsort(mass)[::-1]
    attn = [{"feature": names[i], "mass": round(float(mass[i]), 4)} for i in order[:10]]
    print(json.dumps({"top10_cls_attention": attn}, indent=2))
    with open("weights_v4/classification_pytorch_attn.json", "w") as f:
        json.dump({"top10_cls_attention": attn, "all": {k: round(float(v),5) for k, v in zip(names, mass)}}, f, indent=2)


{
  "top10_cls_attention": [
    {
      "feature": "Recency",
      "mass": 0.1183
    },
    {
      "feature": "Customer_Tenure_Days",
      "mass": 0.092
    },
    {
      "feature": "NumCatalogPurchases",
      "mass": 0.0847
    },
    {
      "feature": "Income",
      "mass": 0.0643
    },
    {
      "feature": "Age",
      "mass": 0.058
    },
    {
      "feature": "Marital_Status_Married",
      "mass": 0.0574
    },
    {
      "feature": "NumWebPurchases",
      "mass": 0.0541
    },
    {
      "feature": "NumStorePurchases",
      "mass": 0.0483
    },
    {
      "feature": "MntFruits",
      "mass": 0.0436
    },
    {
      "feature": "MntWines",
      "mass": 0.0405
    }
  ]
}


## Architecture graph (ONNX — source of the REPORT.md image)
One call to the shared `onnx_viz.save_graph` helper exports the trained oracle above to `media/arch_clf_v4.onnx`
(open it in Netron to inspect) and renders `media/arch_clf_v4.png` (input → … → output).

In [8]:
from onnx_viz import save_graph
save_graph(fresh, "arch_clf_v4", 22)


wrote media/arch_clf_v4.onnx + media/arch_clf_v4.png


'media/arch_clf_v4.onnx'